# Telco Customer Churn - 02. Preprocessing & Feature Engineering
Steps 8-10: preprocessing, feature engineering, class imbalance.

## 0. Setup (recap)
Reloads the data and re-derives earlier decisions, so this notebook can run on its own.

In [1]:
import os, hashlib, warnings, platform, time
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
from scipy import stats
from IPython.display import display

import sklearn
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, cross_val_predict, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler, FunctionTransformer
from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import (roc_auc_score, average_precision_score, precision_score, recall_score,
                             f1_score, confusion_matrix, ConfusionMatrixDisplay, accuracy_score,
                             balanced_accuracy_score, RocCurveDisplay, PrecisionRecallDisplay, brier_score_loss)
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.base import clone
from scipy.stats import loguniform, randint

warnings.filterwarnings("ignore")
SEED = 42
np.random.seed(SEED)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 50, "display.width", 160)

In [2]:
from google.colab import files

uploaded = files.upload()
EXPECTED_FILE = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
assert EXPECTED_FILE in uploaded, f"Please upload {EXPECTED_FILE}"

DATA_PATH = Path(EXPECTED_FILE)
sha = hashlib.sha256(open(DATA_PATH, "rb").read()).hexdigest()
raw = pd.read_csv(DATA_PATH)

ADDONS = ["OnlineSecurity","OnlineBackup","DeviceProtection","TechSupport","StreamingTV","StreamingMovies"]
df0 = raw.copy()
df0["TotalCharges"] = pd.to_numeric(df0.TotalCharges.astype(str).str.strip().replace("", np.nan), errors="coerce").fillna(0.0)
df0["ChurnFlag"] = (df0.Churn == "Yes").astype(int)

y = df0["ChurnFlag"]
X = df0.drop(columns=["customerID", "Churn", "ChurnFlag"])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=SEED)
print("Recap loaded. Train:", X_train.shape, "Test:", X_test.shape, "| SHA-256:", sha)

Recap loaded. Train: (5634, 19) Test: (1409, 19) | SHA-256: 88be4b93fbe0cc83421af1c503794c97c342eca914c1576db7c276e61d61358a


## 8. Data Preprocessing
Build a leak-free pipeline: fix TotalCharges, drop the ID column, simplify redundant categories, encode, and scale.

In [3]:
SIMPLIFY = ["MultipleLines"] + ADDONS
NUM_BASE = ["SeniorCitizen", "tenure", "MonthlyCharges", "TotalCharges"]
CAT_BASE = ["gender","Partner","Dependents","PhoneService","MultipleLines","InternetService"] + ADDONS + \
           ["Contract","PaperlessBilling","PaymentMethod"]

def make_features(X, extras=(), drop_gender=False):
    X = X.copy()
    for c in SIMPLIFY:
        X[c] = X[c].replace({"No internet service": "No", "No phone service": "No"})
    if "n_addons" in extras:
        X["n_addons"] = (X[ADDONS] == "Yes").sum(axis=1)
    if "autopay" in extras:
        X["autopay"] = X["PaymentMethod"].str.contains("automatic").astype(int)
    if "charge_gap" in extras:
        t = X["tenure"].where(X["tenure"] > 0)
        X["charge_gap"] = (X["MonthlyCharges"] - X["TotalCharges"] / t).fillna(0.0)
    if drop_gender:
        X = X.drop(columns=["gender"])
    return X

In [4]:
def build_pipeline(clf, extras=(), drop_gender=False, scale=True):
    num = NUM_BASE + [e for e in ("n_addons", "charge_gap", "autopay") if e in extras]
    cat = [c for c in CAT_BASE if not (drop_gender and c == "gender")]
    prep = ColumnTransformer(
        [("num", StandardScaler() if scale else "passthrough", num),
         ("cat", OneHotEncoder(drop="if_binary", handle_unknown="ignore", sparse_output=False), cat)],
        remainder="drop", verbose_feature_names_out=False)
    steps = [("fe", FunctionTransformer(make_features, kw_args={"extras": tuple(extras), "drop_gender": drop_gender})),
             ("prep", prep), ("clf", clf)]
    return Pipeline(steps)

CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
INNER = StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED)
SCORING = {"roc_auc": "roc_auc", "pr_auc": "average_precision", "precision": "precision", "recall": "recall", "f1": "f1"}
def run_cv(model, X, y, scoring=SCORING):
    res = cross_validate(model, X, y, cv=CV, scoring=scoring, n_jobs=-1)
    return {k[5:]: v for k, v in res.items() if k.startswith("test_")}
def ms(v): return f"{np.mean(v):.3f} ± {np.std(v):.3f}"
print("Pipeline utilities ready.")

Pipeline utilities ready.


## 9. Feature Engineering
Test candidate features (n_addons, autopay, charge_gap) and whether dropping gender costs anything.

In [5]:
configs = {"base": ((), False), "+n_addons": (("n_addons",), False), "+autopay": (("autopay",), False),
           "+charge_gap": (("charge_gap",), False), "drop gender": ((), True)}
two = {"LogReg": (lambda: LogisticRegression(max_iter=2000), True),
       "HistGB": (lambda: HistGradientBoostingClassifier(random_state=SEED, max_iter=150, learning_rate=0.05), False)}

abl = {}
for cname, (ex, dg) in configs.items():
    for mname, (mk, sc) in two.items():
        abl[(cname, mname)] = run_cv(build_pipeline(mk(), ex, dg, sc), X_train, y_train, {"pr_auc": "average_precision"})

rows = []
for cname in configs:
    r = {"config": cname}
    for mname in two:
        r[f"{mname} PR-AUC"] = ms(abl[(cname, mname)]["pr_auc"])
    rows.append(r)
display(pd.DataFrame(rows))

,config,LogReg PR-AUC,HistGB PR-AUC
0,base,0.661 ± 0.019,0.648 ± 0.026
1,+n_addons,0.661 ± 0.019,0.649 ± 0.023
2,+autopay,0.661 ± 0.019,0.652 ± 0.025
3,+charge_gap,0.659 ± 0.019,0.643 ± 0.026
4,drop gender,0.661 ± 0.019,0.650 ± 0.025


In [6]:
EXTRAS = tuple(e for e, k in [("n_addons", "+n_addons"), ("autopay", "+autopay"), ("charge_gap", "+charge_gap")]
               if all(abl[(k, m)]["pr_auc"].mean() > abl[("base", m)]["pr_auc"].mean() for m in two))
DROP_GENDER = all(abl[("drop gender", m)]["pr_auc"].mean() >= abl[("base", m)]["pr_auc"].mean() - 0.001 for m in two)
print("Kept engineered features:", EXTRAS)
print("Drop gender:", DROP_GENDER)

Kept engineered features: ('n_addons',)
Drop gender: True


## 10. Handle Class Imbalance
Compare no reweighting vs class_weight='balanced'.

In [7]:
variants = {
  "no reweighting": build_pipeline(LogisticRegression(max_iter=2000), EXTRAS, DROP_GENDER, True),
  "class_weight=balanced": build_pipeline(LogisticRegression(max_iter=2000, class_weight="balanced"), EXTRAS, DROP_GENDER, True),
}
imb = {n: run_cv(m, X_train, y_train) for n, m in variants.items()}
display(pd.DataFrame({n: {k: ms(imb[n][k]) for k in ["roc_auc","pr_auc","recall","precision","f1"]} for n in imb}).T)

,roc_auc,pr_auc,recall,precision,f1
no reweighting,0.846 ± 0.013,0.661 ± 0.019,0.544 ± 0.041,0.655 ± 0.029,0.594 ± 0.031
class_weight=balanced,0.846 ± 0.012,0.659 ± 0.019,0.803 ± 0.037,0.519 ± 0.019,0.630 ± 0.022


In [8]:
base_n, w_n = "no reweighting", "class_weight=balanced"
USE_WEIGHTS = imb[w_n]["pr_auc"].mean() > imb[base_n]["pr_auc"].mean() + imb[base_n]["pr_auc"].std()
CLASS_WEIGHT = "balanced" if USE_WEIGHTS else None
print("Class weight used in final models:", CLASS_WEIGHT)

Class weight used in final models: None
